In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

Mounted at /content/drive


In [2]:
FILE_PATH = '/content/drive/MyDrive/DVA_Project/data/interim/stat_ready.csv'

df = pd.read_csv(FILE_PATH)

print(df.shape)
df.head()

(11971, 13)


,transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied,year,month
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True,2024,4
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True,2023,7
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False,2022,10
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,False,2022,5
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False,2022,10


In [3]:
df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce')

numeric_cols = ['price_per_unit', 'quantity', 'total_spent']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [4]:
df['year'] = df['transaction_date'].dt.year
df['month'] = df['transaction_date'].dt.month
df['year_month'] = df['transaction_date'].dt.to_period('M').astype(str)

In [5]:
total_revenue = df['total_spent'].sum()
total_transactions = df.shape[0]
avg_order_value = df['total_spent'].mean()
total_quantity = df['quantity'].sum()
unique_customers = df['customer_id'].nunique()

print("Total Revenue:", total_revenue)
print("Total Transactions:", total_transactions)
print("AOV:", avg_order_value)
print("Total Quantity:", total_quantity)
print("Unique Customers:", unique_customers)

Total Revenue: 1552071.0
Total Transactions: 11971
AOV: 129.6525770612313
Total Quantity: 66276.0
Unique Customers: 25


In [6]:
monthly_kpis = df.groupby('year_month').agg({
    'total_spent': ['sum', 'mean'],
    'transaction_id': 'count'
}).reset_index()

monthly_kpis.columns = ['year_month', 'total_revenue', 'avg_order_value', 'transactions']

monthly_kpis.head()

,year_month,total_revenue,avg_order_value,transactions
0,2022-01,52911.5,143.781250,368
1,2022-02,43325.5,138.420128,313
2,2022-03,40996.0,120.932153,339
3,2022-04,40442.0,121.083832,334
4,2022-05,40347.5,128.087302,315


In [7]:
category_metrics = df.groupby('category').agg({
    'total_spent': 'sum',
    'transaction_id': 'count',
    'quantity': 'sum'
}).reset_index()

category_metrics.columns = ['category', 'total_revenue', 'transactions', 'total_quantity']

category_metrics.sort_values(by='total_revenue', ascending=False).head()

,category,total_revenue,transactions,total_quantity
1,Butchers,208118.0,1496,8206.0
3,Electric Household Essentials,203813.5,1516,8309.0
0,Beverages,197047.5,1496,8358.0
5,Furniture,195310.0,1525,8462.0
4,Food,194812.0,1507,8387.0


In [8]:
payment_metrics = df.groupby('payment_method').agg({
    'total_spent': 'sum',
    'transaction_id': 'count'
}).reset_index()

payment_metrics.columns = ['payment_method', 'total_revenue', 'transactions']

payment_metrics

,payment_method,total_revenue,transactions
0,Cash,537710.0,4103
1,Credit Card,507082.0,3927
2,Digital Wallet,507279.0,3941


In [9]:
location_metrics = df.groupby('location').agg({
    'total_spent': 'sum',
    'transaction_id': 'count'
}).reset_index()

location_metrics.columns = ['location', 'total_revenue', 'transactions']

location_metrics

,location,total_revenue,transactions
0,In-Store,760670.0,5903
1,Online,791401.0,6068


In [10]:
discount_metrics = df.groupby('discount_applied').agg({
    'total_spent': ['mean', 'sum'],
    'transaction_id': 'count'
}).reset_index()

discount_metrics.columns = ['discount_applied', 'avg_spent', 'total_revenue', 'transactions']

discount_metrics

,discount_applied,avg_spent,total_revenue,transactions
0,False,129.228810,1027627.5,7952
1,True,130.491043,524443.5,4019


In [11]:
customer_metrics = df.groupby('customer_id').agg({
    'total_spent': 'sum',
    'transaction_id': 'count'
}).reset_index()

customer_metrics.columns = ['customer_id', 'total_spend', 'transactions']

customer_metrics.head()

,customer_id,total_spend,transactions
0,CUST_01,58731.5,485
1,CUST_02,62046.5,467
2,CUST_03,60811.0,446
3,CUST_04,61767.5,455
4,CUST_05,66974.5,516


In [12]:
BASE_PATH = '/content/drive/MyDrive/DVA_Project/data/processed/'

import os
os.makedirs(BASE_PATH, exist_ok=True)

monthly_kpis.to_csv(BASE_PATH + 'monthly_kpis.csv', index=False)
category_metrics.to_csv(BASE_PATH + 'category_metrics.csv', index=False)
payment_metrics.to_csv(BASE_PATH + 'payment_metrics.csv', index=False)
location_metrics.to_csv(BASE_PATH + 'location_metrics.csv', index=False)
discount_metrics.to_csv(BASE_PATH + 'discount_metrics.csv', index=False)
customer_metrics.to_csv(BASE_PATH + 'customer_metrics.csv', index=False)

print("All files saved.")

All files saved.
